# DeepWeeds Lab Day 2 — Kaggle notebook

Enable **Internet** for data, source code, and pretrained weights, and select a **GPU accelerator**. The setup cell uses an attached `kaggle_code_bundle.zip` if present; otherwise it downloads the bundle from this public GitHub repository. Training uses `torch.nn.DataParallel` whenever CUDA is available.

The notebook covers Steps 0–5. All selection and calibration use validation data; Step 4 is the only stage that evaluates test predictions.


## 0. Kaggle setup and repository

No ZIP upload is needed when Internet is enabled. The setup cell downloads the source bundle from GitHub if it is not attached as an input dataset.


In [ ]:
%pip -q install timm==1.0.29 fvcore openpyxl

# Prefer an attached source bundle; otherwise download the current bundle from GitHub.
import os, sys, platform, zipfile, hashlib, subprocess, json
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

REPO_NAME = "K4-Track4-Day2-Deeplearning-Advance"
REPO_DIR = Path("/kaggle/working") / REPO_NAME
REPO_DIR.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = Path("/kaggle/input")
bundle_matches = list(INPUT_ROOT.rglob("kaggle_code_bundle.zip")) if INPUT_ROOT.exists() else []
if len(bundle_matches) > 1:
    raise RuntimeError(f"Found multiple kaggle_code_bundle.zip files: {bundle_matches}")
if bundle_matches:
    BUNDLE_PATH = bundle_matches[0]
else:
    # Kaggle Internet is already required for the data and pretrained weights.
    # Fetch the current companion bundle automatically if it was not added as an input.
    import urllib.request
    BUNDLE_PATH = Path("/kaggle/working/kaggle_code_bundle.zip")
    BUNDLE_URL = "https://raw.githubusercontent.com/vuog23/K4-Track4-Day2-Deeplearning-Advance/main/kaggle_code_bundle.zip"
    print("No bundle attached; downloading the source bundle from GitHub...")
    urllib.request.urlretrieve(BUNDLE_URL, BUNDLE_PATH)
with zipfile.ZipFile(BUNDLE_PATH) as bundle:
    bundle.extractall(REPO_DIR)
os.chdir(REPO_DIR)
CODE_DIR = REPO_DIR / "submissions" / "_draft_vuong" / "code"
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(CODE_DIR))

print("Python", platform.python_version(), "| PyTorch", torch.__version__, "| timm", timm.__version__)
print("GPU(s):", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")
from model import use_data_parallel
print("Training wrapper: torch.nn.DataParallel across", torch.cuda.device_count(), "CUDA device(s)")


## Download and verify DeepWeeds data

The archive extracts the JPG files directly under `data/`; fold-0 CSVs are downloaded unchanged from the dataset author's repository.

In [ ]:
DATA_DIR = REPO_DIR / "data"
LABELS_DIR = DATA_DIR / "labels"
DATA_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / "images.zip"
if not ZIP_PATH.exists():
    subprocess.run(["wget", "-q", "-O", str(ZIP_PATH),
                    "https://zenodo.org/records/7939060/files/images.zip?download=1"], check=True)
md5 = hashlib.md5()
with ZIP_PATH.open("rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        md5.update(chunk)
assert md5.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"images.zip MD5 mismatch: {md5.hexdigest()}"
subprocess.run(["unzip", "-q", "-n", str(ZIP_PATH), "-d", str(DATA_DIR)], check=True)
labels_base = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ("labels", "train_subset0", "val_subset0", "test_subset0"):
    target = LABELS_DIR / f"{name}.csv"
    if not target.exists():
        subprocess.run(["wget", "-q", "-O", str(target), f"{labels_base}/{name}.csv"], check=True)
IMAGES_DIR = str(DATA_DIR)
LABELS_DIR = str(LABELS_DIR)
print("MD5 verified; JPG count:", len(list(DATA_DIR.glob("*.jpg"))))
print("Label CSVs:", sorted(p.name for p in Path(LABELS_DIR).glob("*.csv")))


## Step 0 — EDA and pipeline checks

In [ ]:
from step0 import make_eda, pipeline_smoke
import json
step0_report = make_eda(IMAGES_DIR, LABELS_DIR, out_dir="eda", samples_per_class=3)
smoke_report = pipeline_smoke(IMAGES_DIR, LABELS_DIR)
Path("eda/pipeline_smoke.json").write_text(json.dumps(smoke_report, indent=2), encoding="utf-8")
print("Step 0 complete:", step0_report["n"], smoke_report)


## Shared experiment budget

The notebook uses the lab minimum of 10 epochs, 224-pixel inputs, mixed precision, and batch size 8. If Kaggle reports an out-of-memory error, lower `BATCH_SIZE` (for example to 4) and keep it fixed for comparisons; record that change. `DataParallel` automatically uses every visible GPU.

In [ ]:
from experiments import run_backbones, run_training_ablation
from train import Config, run
from inference_study import run_inference_study
from artifacts import build_results

EPOCHS = 10
BATCH_SIZE = 8
IMG_SIZE = 224
NUM_WORKERS = 0
RUN_DIR = "runs"
PRED_DIR = "predictions"
RESULTS_DIR = "results"
Path(RESULTS_DIR).mkdir(exist_ok=True)


## GPU budget pilot

This first runs one epoch per planned backbone (validation only), saves a checkpoint/log after each epoch, and estimates the 20-run budget from measured times. The five backbones are ResNet-50, ResNeXt-50, DeiT-S, ResNet-18, and RegNetY-400MF. This includes the required ResNet, ResNeXt, transformer, and lightweight families. ResNet-18 also passed the Step 0 pipeline smoke check. RegNetY-400MF is a compact model (4.3M parameters, 0.4 GMAC at 224px) and is included instead of another MBConv-based model after the EfficientNet pilot stalled. The pilot uses batch size 8 and no DataLoader worker processes to reduce Kaggle GPU/RAM pressure; `DataParallel` still uses every visible GPU. If the kernel restarts while `/kaggle/working` is still intact, rerun the setup/data cells and then this cell: completed `Qxx` runs with matching settings are reused, and only incomplete runs restart. Review the estimate before starting the long experiment cells. The pilot uses separate `Qxx` IDs and does not touch test data.


In [ ]:
from experiments import run_budget_pilot
budget_results = run_budget_pilot(
    batch_size=BATCH_SIZE, img_size=IMG_SIZE, epochs=EPOCHS,
    out_csv=f"{RESULTS_DIR}/budget_pilot.csv", out_dir="runs_budget",
    pred_dir="predictions_budget", num_workers=NUM_WORKERS)
budget_results


## Step 1 — Compare at least five backbones

Each backbone uses the same T00 recipe, fold, seed, epoch count, resolution, and batch size. Checkpoint selection uses validation macro-F1.

In [ ]:
backbone_results = run_backbones(
    out_csv=f"{RESULTS_DIR}/backbones.csv", batch_size=BATCH_SIZE, epochs=EPOCHS,
    img_size=IMG_SIZE, out_dir=RUN_DIR, pred_dir=PRED_DIR, num_workers=NUM_WORKERS)
backbone_results.sort_values("macro_f1_val", ascending=False)


## Step 2 — Training recipe ablations

The runner compares T00 with one-factor changes and a combined candidate. It chooses the backbone only from Step 1 validation macro-F1; test data is not loaded.

In [ ]:
selected_backbone = str(backbone_results.sort_values("macro_f1_val", ascending=False).iloc[0]["backbone"])
print("Selected by validation macro-F1:", selected_backbone)
training_results = run_training_ablation(
    selected_backbone, out_csv=f"{RESULTS_DIR}/training.csv", batch_size=BATCH_SIZE,
    epochs=EPOCHS, img_size=IMG_SIZE, out_dir=RUN_DIR, pred_dir=PRED_DIR,
    num_workers=NUM_WORKERS)
training_results.sort_values("macro_f1_val", ascending=False)


## Step 3 — Validation inference and latency

This compares one-view inference, flips/crops, resolution, calibration, and latency. Temperature is fitted on validation only. The model is not retrained in this step.

In [ ]:
best_training = training_results.sort_values("macro_f1_val", ascending=False).iloc[0]
best_exp_id = str(best_training["exp_id"])
best_backbone = str(best_training["backbone"])
inference_results, temperature = run_inference_study(
    best_exp_id, best_backbone, out_csv=f"{RESULTS_DIR}/inference.csv",
    batch_size=BATCH_SIZE, img_size=IMG_SIZE, run_root=RUN_DIR,
    images_dir=IMAGES_DIR, labels_dir=LABELS_DIR)
print("Best recipe:", best_exp_id, best_backbone, "| validation temperature:", temperature)
inference_results


## Step 4 — Final comparison (test is used only here)

Run this section once after reviewing Steps 1–3 and fixing the recipe. It retrains the chosen recipe and T00 baseline for seeds 0, 1, 2 and writes one test prediction file per run. Do not change the model after examining these test results.

In [ ]:
# Freeze the validation-selected training recipe. F01 uses I00 one-view inference plus
# temperature scaling fitted on that seed's validation logits; T00 remains the uncalibrated I00 baseline.
selected = training_results.sort_values("macro_f1_val", ascending=False).iloc[0]
final_backbone = str(selected["backbone"])
raw_changes = selected.get("changed_from_T00", "{}")
final_recipe = json.loads(raw_changes) if isinstance(raw_changes, str) else {}
print("Frozen final choice from validation:", final_backbone, final_recipe)

# This is the only cell that enables test prediction. run_final_comparison evaluates test once,
# saves its logits, then writes calibrated and uncalibrated CSVs without another test forward pass.
from experiments import run_final_comparison
final_runs = run_final_comparison(
    final_backbone, final_recipe=final_recipe, seeds=(0, 1, 2),
    batch_size=BATCH_SIZE, epochs=EPOCHS, img_size=IMG_SIZE,
    out_dir=RUN_DIR, pred_dir=PRED_DIR, num_workers=NUM_WORKERS,
    labels_dir=str(LABELS_DIR))
final_runs


In [ ]:
# Score the already-saved prediction CSVs; this does not run the model on test again.
# I00 was benchmarked with warmup, CUDA synchronization, and 50 timed batch-1 calls.
i00_latency = inference_results.loc[inference_results["exp_id"] == "I00", "p95"].iloc[0]
subprocess.run([sys.executable, str(REPO_DIR / "eval.py"), "score", "--pred",
                f"{PRED_DIR}/F01_seed*_test.csv", "--test-csv",
                f"{LABELS_DIR}/test_subset0.csv", "--labels",
                f"{LABELS_DIR}/labels.csv", "--tag", "F01", "--out", "eval_out"], check=True)
subprocess.run([sys.executable, str(REPO_DIR / "eval.py"), "score", "--pred",
                f"{PRED_DIR}/T00_seed*_test.csv", "--test-csv",
                f"{LABELS_DIR}/test_subset0.csv", "--labels",
                f"{LABELS_DIR}/labels.csv", "--tag", "T00", "--out", "eval_out"], check=True)
subprocess.run([sys.executable, str(REPO_DIR / "eval.py"), "grade", "--final",
                f"{PRED_DIR}/F01_seed*_test.csv", "--baseline",
                f"{PRED_DIR}/T00_seed*_test.csv", "--uncal",
                f"{PRED_DIR}/F01uncal_seed*_test.csv", "--final-val",
                f"{PRED_DIR}/F01_seed*_val.csv", "--val-csv",
                f"{LABELS_DIR}/val_subset0.csv", "--test-csv",
                f"{LABELS_DIR}/test_subset0.csv", "--labels",
                f"{LABELS_DIR}/labels.csv", "--latency-p95-ms", str(float(i00_latency)),
                "--latency-method", "proper", "--out", "eval_out"], check=True)


## Step 5 — Workbook and report

In [ ]:
xlsx_path, report_path = build_results(
    results_dir=RESULTS_DIR, eval_dir="eval_out",
    out_xlsx="results.xlsx", out_report="report.md",
    pred_dir=PRED_DIR, labels_dir=str(LABELS_DIR), images_dir=IMAGES_DIR)
print("Created:", xlsx_path.resolve(), report_path.resolve())
print("Workbook sheets: Backbones, Training, Inference, Final, PerClass, Latency, Summary")
